---
numbering: false
---

# 2.4: Lines and planes through the origin in ℝ³


In [1]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import sys

_notes_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'myst.yml').exists())
if str(_notes_root) not in sys.path:
    sys.path.insert(0, str(_notes_root))
from plot_style import style_plotly

BLUE, ORANGE, PINK = '#3d81f6', 'orange', '#d81a60'


def base3():
    fig=style_plotly(go.Figure(),renderer='plotly_mimetype')
    axis=dict(range=[-6,8],dtick=2,tickfont=dict(size=11),showbackground=True,showspikes=False,
              backgroundcolor='white',gridcolor='#e5e7eb',zerolinecolor='#9ca3af')
    fig.update_layout(autosize=True,height=520,showlegend=False,font=dict(size=16),
                      margin=dict(l=0,r=0,t=15,b=0),
                      scene=dict(bgcolor='white',xaxis=dict(title=dict(text='x',font=dict(size=13)),**axis),yaxis=dict(title=dict(text='y',font=dict(size=13)),**axis),
                                 zaxis=dict(title=dict(text='z',font=dict(size=13)),**axis),aspectmode='cube',
                                 camera=dict(eye=dict(x=1.6,y=-2.1,z=1.3))))
    for direction in np.eye(3):
        line3(fig,-6*direction,8*direction,'#9ca3af',width=2)
    return fig


def line3(fig,start,end,color=BLUE,width=5,dash='solid'):
    fig.add_trace(go.Scatter3d(x=[start[0],end[0]],y=[start[1],end[1]],z=[start[2],end[2]],
                              mode='lines',line=dict(color=color,width=width,dash=dash),
                              hoverinfo='skip',showlegend=False))


def vec3(fig,end,label,color=BLUE,start=(0,0,0),offset=(0.25,0.25,0.35)):
    start,end=np.asarray(start,float),np.asarray(end,float)
    direction=(end-start)/np.linalg.norm(end-start)
    line3(fig,start,end-0.15*direction,color,width=7)
    fig.add_trace(go.Cone(x=[end[0]],y=[end[1]],z=[end[2]],u=[direction[0]],v=[direction[1]],w=[direction[2]],
                         anchor='tip',sizemode='absolute',sizeref=0.45,colorscale=[[0,color],[1,color]],
                         showscale=False,hoverinfo='skip'))
    pos=end+offset
    fig.add_trace(go.Scatter3d(x=[pos[0]],y=[pos[1]],z=[pos[2]],mode='text',text=[label],
                              textfont=dict(family='Palatino',color=color,size=20),hoverinfo='skip'))


def plane3(fig,normal,color=BLUE,opacity=0.24,d=0,bounds=(-5,7)):
    # Solve n_x*x + n_y*y + n_z*z = d for y on a rectangular x,z mesh.
    x,z=np.meshgrid(np.linspace(*bounds,2),np.linspace(*bounds,2))
    n=np.asarray(normal,float)
    y=(d-n[0]*x-n[2]*z)/n[1]
    fig.add_trace(go.Surface(x=x.tolist(),y=y.tolist(),z=z.tolist(),
                            colorscale=[[0,color],[1,color]],opacity=opacity,
                            showscale=False,hoverinfo='skip'))
    corners=np.array([[x[0,0],y[0,0],z[0,0]],[x[0,1],y[0,1],z[0,1]],
                      [x[1,1],y[1,1],z[1,1]],[x[1,0],y[1,0],z[1,0]]])
    for i in range(4): line3(fig,corners[i],corners[(i+1)%4],color,width=2)




In [Chapter 2.1](02-01.ipynb), we learned how to describe a line in $\mathbb R^2$ using scalar multiples of a nonzero vector. We'll use the same idea here, with one extra coordinate, and then describe planes using linear combinations of two vectors.

```{attention} Note: Chapter 2.4 has been split into three pages
- [Chapter 2.4](02-04.ipynb) now introduces lines and planes in $\mathbb R^3$ and the concept of the **span** of one or two vectors in $\mathbb R^3$.
- [Chapter 2.5](02-05.ipynb) details homogeneous linear equations – equations of the form

  $$ax + by + cz = 0$$

  – and what they represent in $\mathbb R^3$.
- [Chapter 2.6](02-06.ipynb) talks about **affine** lines and planes – that is, lines and planes that don't need to pass through the origin.
```

This section, along with the rest of Chapter 2, contains several 3D figures. Drag them with your mouse to view the lines and planes from different perspectives.



---

## Lines through the origin

Remember that the span of one vector is the set of all its scalar multiples. Let's start with

$$\vec v=\begin{bmatrix}3\\4\\5\end{bmatrix}.$$

Multiplying $\vec v$ by a scalar changes its length and possibly reverses its direction. If we draw all of these multiples from the origin, their tips trace out a line! As before, we can write this line as

$$\ell=\operatorname{span}(\vec v)
=\left\{t\begin{bmatrix}3\\4\\5\end{bmatrix}:t\in\mathbb R\right\}.$$



In [2]:
fig=base3()
v=np.array([3,4,5])
line3(fig,-1.1*v,1.3*v,BLUE,width=4)
vec3(fig,v,'<i>v</i>⃗',BLUE)
fig.show()


```{figure} #plot-24-line
:label: fig-24-line
:class: course-caption
:alt: A line through the origin with direction vector v.

The line through the origin spanned by
$\vec v=\begin{bmatrix}3\\4\\5\end{bmatrix}$.
```



:::{note} Definition: Vector-parametric and scalar-parametric forms of lines
The **vector-parametric form** of a line describes a line using vectors and a parameter. We can write it using set-builder notation, or as an equation for the position vector of a point. For the line above, these are

$$\ell=\left\{t\begin{bmatrix}3\\4\\5\end{bmatrix}:t\in\mathbb R\right\}$$

and

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=t\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad t\in\mathbb R.$$

We will treat these as equivalent – if we ask you to provide the vector-parametric form of a line, either one is sufficient.

The **scalar-parametric form** of a line has one equation per component:

$$x=3t,\qquad y=4t,\qquad z=5t,\qquad t\in\mathbb R.$$

All three equations use the **same parameter** $t$. Each choice of $t$ gives one point on the line.
:::

For instance, $t=1$ gives $(3,4,5)$, $t=-1$ gives $(-3,-4,-5)$, and $t=0$ gives the origin. The parameter can be any real number; the figure only shows part of the line.

To move from vector form to scalar form, read off the entries. To move back, collect the coefficients of the parameter into a vector.

::::{tip} Activity 1
Let

$$\ell=\operatorname{span}\left(\begin{bmatrix}6\\-7\\11\end{bmatrix}\right).$$

1. Write both vector-parametric forms: one using set-builder notation and one using a position-vector equation. Then write the scalar-parametric form.
2. Give a different vector with the same span. Write both vector-parametric forms using that vector, and explain why the line is unchanged.

:::{tip} Solution
:class: dropdown

Using the given vector, the two vector-parametric forms are

$$\ell=\left\{t\begin{bmatrix}6\\-7\\11\end{bmatrix}:t\in\mathbb R\right\},$$

$$\begin{bmatrix}x\\y\\z\end{bmatrix}=t\begin{bmatrix}6\\-7\\11\end{bmatrix},\qquad t\in\mathbb R.$$

The scalar-parametric form is

$$x=6t,\qquad y=-7t,\qquad z=11t,\qquad t\in\mathbb R.$$

One alternative direction vector is

$$\vec d=\begin{bmatrix}12\\-14\\22\end{bmatrix}=2\begin{bmatrix}6\\-7\\11\end{bmatrix}.$$

It gives the equivalent vector-parametric forms

$$\ell=\left\{s\begin{bmatrix}12\\-14\\22\end{bmatrix}:s\in\mathbb R\right\},$$

$$\begin{bmatrix}x\\y\\z\end{bmatrix}=s\begin{bmatrix}12\\-14\\22\end{bmatrix},\qquad s\in\mathbb R.$$

The spans agree because we can rescale the parameter in either direction:

$$t\begin{bmatrix}6\\-7\\11\end{bmatrix}
=\frac t2\begin{bmatrix}12\\-14\\22\end{bmatrix}.$$
:::
::::



Here's a video by Kartik that visualizes this idea further.

```{iframe} https://www.youtube.com/embed/NZ3Zf6diubo
:width: 100%
:title: The span of a vector is a line
```




---

## Planes through the origin

### Linear independence



To fill a plane through the origin, we need two directions that cannot be obtained by scaling one another. Let's give this condition a name.

:::{note} Definition: Linear independence of two vectors
Two vectors are **linearly independent** if neither is a scalar multiple of the other. Otherwise, they are **linearly dependent**.
:::

We'll often shorten "linearly independent" to just "independent."

For two nonzero vectors, independence means that they point along different lines. For example, $\begin{bmatrix}1\\0\\0\end{bmatrix}$ and $\begin{bmatrix}0\\1\\0\end{bmatrix}$ are independent and span the $xy$-plane. But $\begin{bmatrix}1\\0\\0\end{bmatrix}$ and $\begin{bmatrix}2\\0\\0\end{bmatrix}$ are dependent and span only the $x$-axis. The second vector adds no new direction.

This definition of linear independence is defined for a pair of vectors. In later chapters, we'll discuss what it means for three or more vectors to be linearly independent.


### Span of two vectors

Back to the main idea: describing a plane in $\mathbb{R}^3$.

For example, let

$$\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad
\vec v_2=\begin{bmatrix}5\\2\\-1\end{bmatrix}.$$

These vectors are independent: matching the second entry of $\vec v_1$ would require multiplying $\vec v_2$ by $2$, but that would give a first entry of $10$, not $3$. So, there's no number we can multiply $\vec v_2$ by to get $\vec v_1$.

Notice that we're asking for less than we did in [Chapter 2.2](02-02.ipynb). Our vectors don't need to have length one, and they don't need to be perpendicular. We just need two independent directions in the plane.



In [3]:
fig=base3()
plane3(fig,(1,-2,1))
vec3(fig,(3,4,5),'<i>v</i>⃗<sub>1</sub>',BLUE)
vec3(fig,(5,2,-1),'<i>v</i>⃗<sub>2</sub>',ORANGE,offset=(-0.8,0.2,0.4))
fig.show()


```{figure} #plot-24-span
:label: fig-24-span
:class: course-caption
:alt: Two independent vectors span the blue plane through the origin.

The vectors $\vec v_1$ and $\vec v_2$ determine a plane $P$ through
the origin.
```


For example, consider $2\vec v_1-3\vec v_2$, a linear combination of $\vec v_1$ and $\vec v_2$:

$$2\vec v_1-3\vec v_2
=2\begin{bmatrix}3\\4\\5\end{bmatrix}
-3\begin{bmatrix}5\\2\\-1\end{bmatrix}
=\begin{bmatrix}-9\\2\\13\end{bmatrix}.$$

Draw $2\vec v_1$ from the origin, then draw $-3\vec v_2$ from the tip of $2\vec v_1$. The vector from the origin to the final tip is $2\vec v_1-3\vec v_2$.


In [4]:
fig=base3()
plane3(fig,(1,-2,1),opacity=0.15,bounds=(-11,15))
v1=np.array([3,4,5]); v2=np.array([5,2,-1])
a=2*v1; w=a-3*v2
fig.update_scenes(xaxis_range=[-12,16],yaxis_range=[-12,16],zaxis_range=[-12,16],
                  xaxis_dtick=4,yaxis_dtick=4,zaxis_dtick=4)
for direction in np.eye(3):
    line3(fig,-12*direction,16*direction,'#9ca3af',width=2)
vec3(fig,a,'2<i>v</i>⃗<sub>1</sub>',BLUE,offset=(0.6,0.6,0.7))
vec3(fig,w,'−3<i>v</i>⃗<sub>2</sub>',ORANGE,start=a,offset=(0.6,0.6,0.7))
# Put the translated-vector label at its midpoint so the two final-tip labels do not overlap.
mid=(a+w)/2+np.array([0,0,1])
fig.data[-1].update(x=[mid[0]],y=[mid[1]],z=[mid[2]])
vec3(fig,w,'2<i>v</i>⃗<sub>1</sub> − 3<i>v</i>⃗<sub>2</sub>',PINK,offset=(-0.5,-0.5,1))
fig.show()


```{figure} #plot-24-linear-combination
:label: fig-24-linear-combination
:class: course-caption
:alt: The blue vector 2v1 starts at the origin. The orange vector negative 3v2 starts at its tip. The pink resultant connects the origin to the final tip.

Adding $2\vec v_1$ and $-3\vec v_2$ head to tail gives $2\vec v_1-3\vec v_2$.
```

Notice that $2\vec v_1-3\vec v_2$ also lives on this same plane. In fact, every linear combination of $\vec v_1$ and $\vec v_2$ lives on this plane, and every vector that lives on this plane can be written as a linear combination of $\vec v_1$ and $\vec v_2$!




:::{note} Definition: Span of two vectors
The **span** of $\vec v_1$ and $\vec v_2$ is the set of all their linear combinations:

$$\operatorname{span}(\vec v_1,\vec v_2)
=\{a\vec v_1+b\vec v_2:a,b\in\mathbb R\}.$$

If the two vectors are independent, their span is a plane through the origin.
:::

In our example, this gives

$$\begin{aligned}
P&=\operatorname{span}(\vec v_1,\vec v_2)\\
&=\left\{a\begin{bmatrix}3\\4\\5\end{bmatrix}
+b\begin{bmatrix}5\\2\\-1\end{bmatrix}:a,b\in\mathbb R\right\}\\
&=\left\{\begin{bmatrix}3a+5b\\4a+2b\\5a-b\end{bmatrix}:a,b\in\mathbb R\right\}.
\end{aligned}$$

You can think of $a$ and $b$ as two knobs we can turn: $a$ tells us how much of $\vec v_1$ to use, and $b$ tells us how much of $\vec v_2$ to use. Letting both range over all real numbers gives the entire plane.





### Parametric equations

The same representation terminology applies to planes. Both of the following are **vector-parametric** forms of $P$:

$$P=\left\{a\begin{bmatrix}3\\4\\5\end{bmatrix}
+b\begin{bmatrix}5\\2\\-1\end{bmatrix}:a,b\in\mathbb R\right\},$$

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=a\begin{bmatrix}3\\4\\5\end{bmatrix}
+b\begin{bmatrix}5\\2\\-1\end{bmatrix},\qquad a,b\in\mathbb R.$$

Expanding the vector sum gives

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}3a+5b\\4a+2b\\5a-b\end{bmatrix}.$$

Reading the entries gives the **scalar-parametric form**:

$$x=3a+5b,\qquad y=4a+2b,\qquad z=5a-b,\qquad a,b\in\mathbb R.$$

In the scalar-parametric form above, all three equations use the same two parameters. Each pair $(a,b)$ selects one point; allowing both parameters to range independently over all real numbers fills the plane.

Note that the letters $a$ and $b$ are arbitrary. We could, and often do, use $s$ and $t$ as well.

We can easily move back and forth between the two parametric descriptions of a plane. For example, suppose we're given the scalar-parametric form

$$x=s+t,\qquad y=s-t,\qquad z=2t,\qquad s,t\in\mathbb R.$$

Collecting the coefficients of $s$ and $t$ gives

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=s\begin{bmatrix}1\\1\\0\end{bmatrix}
+t\begin{bmatrix}1\\-1\\2\end{bmatrix},\qquad s,t\in\mathbb R.$$

Equivalently, this plane is

$$\left\{s\begin{bmatrix}1\\1\\0\end{bmatrix}
+t\begin{bmatrix}1\\-1\\2\end{bmatrix}:s,t\in\mathbb R\right\}$$

or even

$$\left\{a\begin{bmatrix}1\\1\\0\end{bmatrix}
+b\begin{bmatrix}1\\-1\\2\end{bmatrix}:a,b\in\mathbb R\right\}.$$

### Different vectors spanning the same plane

Recall, the plane $P$ is defined as the span of $\vec v_1$ and $\vec v_2$,

$$\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad
\vec v_2=\begin{bmatrix}5\\2\\-1\end{bmatrix}.$$

**Key idea**: the exact same plane, $P$, can also be written as the span of other pairs of vectors! To illustrate, let's define two new vectors, $\vec v_3$ and $\vec v_4$, using our original two:

$$\vec v_3=\vec v_1-\vec v_2,\qquad\vec v_4=2\vec v_1+3\vec v_2.$$

The resulting vectors

$$\vec v_3=\begin{bmatrix}-2\\2\\6\end{bmatrix},\qquad
\vec v_4=\begin{bmatrix}21\\14\\7\end{bmatrix},$$

also span $P$:

$$P=\operatorname{span}(\vec v_1,\vec v_2)=\operatorname{span}(\vec v_3,\vec v_4).$$

Why? **Each pair can be built from the other pair**. That means anything we can build using one pair can also be built using the other. Let's check this carefully.

- By construction, any linear combination of $\vec v_3$ and $\vec v_4$ is also a linear combination of $\vec v_1$ and $\vec v_2$:
    $$\begin{aligned}
    c\vec v_3+d\vec v_4
    &=c(\vec v_1-\vec v_2)+d(2\vec v_1+3\vec v_2)\\
    &=(c+2d)\vec v_1+(-c+3d)\vec v_2.
    \end{aligned}$$
    So every vector in $\operatorname{span}(\vec v_3,\vec v_4)$ belongs to $P$.
- But that's only half of what we need. We also need to show that we can build $\vec v_1$ and $\vec v_2$ using $\vec v_3$ and $\vec v_4$. Notice that
    $$\begin{aligned}
    3\vec v_3+\vec v_4
    &=3(\vec v_1-\vec v_2)+(2\vec v_1+3\vec v_2)=5\vec v_1,\\
    \vec v_4-2\vec v_3
    &=(2\vec v_1+3\vec v_2)-2(\vec v_1-\vec v_2)=5\vec v_2
    \end{aligned}$$
    so
    $$\vec v_1=\frac35\vec v_3+\frac15\vec v_4,\qquad
    \vec v_2=-\frac25\vec v_3+\frac15\vec v_4.$$
    Therefore, any vector in $P$ can also be written as
    $$\begin{aligned}
    a\vec v_1+b\vec v_2
    &=a\left(\frac35\vec v_3+\frac15\vec v_4\right)
    +b\left(-\frac25\vec v_3+\frac15\vec v_4\right)\\
    &=\frac{3a-2b}{5}\vec v_3+\frac{a+b}{5}\vec v_4.
    \end{aligned}$$

So, we can move back and forth between the two descriptions: **any linear combination of $\vec v_1$ and $\vec v_2$ is also a linear combination of $\vec v_3$ and $\vec v_4$, and vice versa**. The coefficients change, but the set of vectors we can reach stays the same!

More generally, any two linearly independent vectors **in $P$** span $P$. As in [Chapter 2.1](02-01.ipynb), a span description is not unique.

For our original plane, the replacement vectors above give another vector-parametric form:

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=s\begin{bmatrix}-2\\2\\6\end{bmatrix}
+t\begin{bmatrix}21\\14\\7\end{bmatrix},\qquad s,t\in\mathbb R.$$

Its scalar-parametric form is

$$x=-2s+21t,\qquad y=2s+14t,\qquad z=6s+7t.$$

The point $(3,4,5)$ occurred at $a=1,b=0$ in the original parametrization. Here it occurs at $s=3/5,t=1/5$. **The same point can have different parameter values in different parametrizations.**

::::{tip} Activity 2
Let $\vec u=\begin{bmatrix}1\\1\\0\end{bmatrix}$ and $\vec v=\begin{bmatrix}1\\-1\\2\end{bmatrix}$.

1. Find $\vec u+\vec v$ and $\vec u-\vec v$. Explain why they span the same plane as $\vec u,\vec v$.
2. Use the new pair to write scalar-parametric equations, including the possible values of the parameters.

:::{tip} Solution
:class: dropdown

The new vectors are $\begin{bmatrix}2\\0\\2\end{bmatrix}$ and $\begin{bmatrix}0\\2\\-2\end{bmatrix}$. Each is a linear combination of $\vec u,\vec v$, and we can recover the original pair:

$$\vec u=\tfrac12(\vec u+\vec v)+\tfrac12(\vec u-\vec v),\qquad
\vec v=\tfrac12(\vec u+\vec v)-\tfrac12(\vec u-\vec v).$$

Each pair can be built from the other, so their spans agree. Using the new pair gives

$$x=2s,\qquad y=2t,\qquad z=2s-2t,\qquad s,t\in\mathbb R.$$
:::
::::

Here's a video by Kartik synthesizing the main ideas of this section.

```{iframe} https://www.youtube.com/embed/ELVbshKqEbs
:width: 100%
:title: Different vectors spanning the same plane
```

What's next? This section described how to express lines and planes in $\mathbb{R}^3$ in parametric form, using the fact that they can be thought of as spans of vectors. In [Chapter 2.5](./02-05.ipynb), we will describe lines and planes in $\mathbb{R}^3$ using linear equations: that is, equations of the form

$$ax + by + cz = d.$$